In [2]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:
with open("/kaggle/input/datasets/ronikdedhia/next-word-prediction/1661-0.txt", "r",encoding='utf-8') as f:
    text = f.read()

In [4]:
print(len(text))   # prints length of all the characters
print(text[:500])

581888
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charac


### Text Cleaning

In [5]:
import string
def text_cleaning(txt):
    txt = txt.lower()
    txt = txt.replace('\n',' ').replace('\r',' ')

    cleaned_chars=[]
    for char in txt:
        if char in string.ascii_lowercase or char==' ':
            cleaned_chars.append(char)

    cleaned_text = "".join(cleaned_chars)            # joins all the characters into words
    cleaned_text = " ".join(cleaned_text.split())    # puts one space btwn each word
    
    return cleaned_text

In [6]:
text_cleaning('Hello!!! my name is god,,, i am from world')

'hello my name is god i am from world'

In [7]:
text = text_cleaning(text)

text[:1000]

'project gutenbergs the adventures of sherlock holmes by arthur conan doyle this ebook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever you may copy it give it away or reuse it under the terms of the project gutenberg license included with this ebook or online at wwwgutenbergnet title the adventures of sherlock holmes author arthur conan doyle release date november ebook last updated may language english character set encoding utf start of this project gutenberg ebook the adventures of sherlock holmes produced by an anonymous project gutenberg volunteer and jose menendez cover the adventures of sherlock holmes by arthur conan doyle contents i a scandal in bohemia ii the redheaded league iii a case of identity iv the boscombe valley mystery v the five orange pips vi the man with the twisted lip vii the adventure of the blue carbuncle viii the adventure of the speckled band ix the adventure of the engineers thumb x the adventure of the noble bachelo

In [8]:
a = text.split()
a[:10]

['project',
 'gutenbergs',
 'the',
 'adventures',
 'of',
 'sherlock',
 'holmes',
 'by',
 'arthur',
 'conan']

### counting fequency of the words

In [9]:
words = text.split()

word_counts = {}
for word in words:
    if word in word_counts:
        word_counts[word] = word_counts[word] + 1
    else:
        word_counts[word] = 1

In [10]:
len(word_counts)
print()
print(list(word_counts.items())[:10])


[('project', 90), ('gutenbergs', 1), ('the', 5805), ('adventures', 11), ('of', 2778), ('sherlock', 102), ('holmes', 463), ('by', 374), ('arthur', 21), ('conan', 4)]


### selecting top words from vocab

In [11]:
vocab_size=5000
sorted_words = sorted(word_counts.items(), key=lambda x:x[1], reverse=True)
top_words = [word for word, count in sorted_words[ : vocab_size-1]]

In [12]:
print(sorted_words[:10])
print() 
print(top_words[:10])

[('the', 5805), ('and', 3070), ('i', 2995), ('of', 2778), ('to', 2762), ('a', 2683), ('in', 1818), ('that', 1750), ('it', 1710), ('you', 1545)]

['the', 'and', 'i', 'of', 'to', 'a', 'in', 'that', 'it', 'you']


In [13]:
# mapping words to indices

word2idx = {word: idx + 1 for idx, word in enumerate(top_words)}
word2idx["<UNK>"] = 0
idx2word = ["<UNK>"] + top_words

In [17]:
print(idx2word[:10])
print()
print(list(word2idx)[:10])

['<UNK>', 'the', 'and', 'i', 'of', 'to', 'a', 'in', 'that', 'it']

['the', 'and', 'i', 'of', 'to', 'a', 'in', 'that', 'it', 'you']


In [18]:
token_seq = [word2idx.get(word, 0) for word in words]

In [19]:
sequence_length = 10
input_sequences = []

# Generate sliding window sequences of length sequence_length + 1
for i in range(sequence_length, len(token_seq)):
    input_sequences.append(token_seq[i - sequence_length : i + 1])
    
# Split into inputs (X) and targets (y)
input_sequences = np.array(input_sequences)
x = input_sequences[:, :-1]
y = input_sequences[:, -1]



In [20]:
print("X shape:", x.shape)
print("y shape:", y.shape)

X shape: (107420, 10)
y shape: (107420,)
